# LSTM: selección de variables por bloques

Primera parte de la LSTM nueva, según `docs/LSTM_DISENO.md`. Pronostica la semana siguiente hora
por hora (h = 3 a 9, 168 precios) con un codificador-decodificador, y decide con la misma red qué
bloques de variables entran.

1. Selección hacia adelante de los bloques 3 a 6, con la arquitectura fija.
2. Prueba de control con todas las variables e importancia por grupo.
3. Búsqueda corta de control, para comprobar que la configuración fija no sesga la selección.
4. Si la búsqueda encuentra una configuración que le gana a la fija, la selección se repite con ella.

Todo se decide con 2015-2022. Este notebook no lee nada de 2023 en adelante: el test se corre
después, una sola vez, con el modelo ya cerrado.

Se corre en Colab con GPU. Hay que subir `panel_lstm_bloques.csv.gz` y
`panel_lstm_bloques_variables.csv`, que arma `2h_panel_lstm_bloques.py`. Cada pronóstico se guarda
apenas termina, así que si la sesión se cae basta volver a correr todo y sigue donde iba.

In [ ]:
import json
from math import erf, sqrt
from pathlib import Path

import keras
import numpy as np
import pandas as pd
from keras import layers

# True: corre todo en pocos minutos con una red mínima, solo para revisar que no falle nada
PRUEBA_RAPIDA = False

RUTA = Path("paneles/panel_lstm_bloques.csv.gz")
if not RUTA.exists():
    RUTA = Path("panel_lstm_bloques.csv.gz")
RUTA_VARIABLES = RUTA.with_name("panel_lstm_bloques_variables.csv")

SALIDA = Path("resultados/modelo_lstm_bloques")
# Para no perder lo corrido si se cae la sesión de Colab, guardar en Drive:
# from google.colab import drive
# drive.mount("/content/drive")
# SALIDA = Path("/content/drive/MyDrive/tesis/modelo_lstm_bloques")

HS = list(range(3, 10))
HS_PRUEBA = [3, 6, 9]              # mañana, mitad y final de la semana
INICIO = "2015-03-01"
TECHO = "2022-12-31"               # nada de este notebook pasa de aquí
AÑOS = [2020, 2021, 2022]          # años de validación; cada uno se entrena con todo lo anterior
SEMILLAS = [0, 1, 2]
NIVEL = 0.05
VENTANA_MAX = 56

FIJA = dict(capas=1, unidades=128, densa=64, dropout=0.2, tasa=1e-3, lote=32, ventana=28)
EPOCAS, PACIENCIA = 100, 10
N_BUSQUEDA = 10

if PRUEBA_RAPIDA:
    SALIDA = SALIDA / "prueba_rapida"
    SEMILLAS, EPOCAS, N_BUSQUEDA = [0], 2, 2
    FIJA = dict(FIJA, unidades=16, densa=16)

(SALIDA / "pronosticos").mkdir(parents=True, exist_ok=True)
print("keras", keras.__version__, "con", keras.backend.backend())

## 1. Datos

Cada fila del panel (día t, hora H) tiene lo que se sabe el día del pronóstico, R = t + 2 (paso
2e). De ahí salen tres arreglos, con un elemento por día:

- la historia: un vector por día con las 24 horas del precio, las 24 de cada variable horaria, una
  vez cada variable diaria y el calendario de ese día;
- el calendario de los 7 días objetivo (día de la semana y festivo);
- el objetivo: `y_h` = lp(t + h, H) − lp(t, H) para h = 3 a 9, el mismo de XGBoost y SARIMAX.

In [ ]:
info = pd.read_csv(RUTA_VARIABLES).set_index("variable")
panel = pd.read_csv(RUTA, parse_dates=["fecha"]).sort_values(["fecha", "hora"]).reset_index(drop=True)
assert (panel.groupby("fecha").size() == 24).all()

f = pd.DatetimeIndex(panel["fecha"].drop_duplicates())
n_dias = len(f)
dia = panel.iloc[::24].reset_index(drop=True)


def por_dia(col):
    return panel[col].to_numpy(np.float32).reshape(n_dias, 24)


def calendario(dia_semana, festivo):
    return np.column_stack([np.eye(7, dtype=np.float32)[dia_semana], festivo.astype(np.float32)])


LP = por_dia("lp")
Y = np.stack([por_dia(f"y_h{h}") for h in HS], axis=1)                       # [día, 7, 24]
CAL_T = calendario(dia["dia_semana"].to_numpy(), dia["es_festivo"].to_numpy())  # [día, 8]
CAL_K = np.stack([calendario(dia[f"dia_semana_k{k}"].to_numpy(), dia[f"es_festivo_k{k}"].to_numpy())
                  for k in range(1, 8)], axis=1)                              # [día, 7, 8]

NOMBRES = info[info["bloque"] > 0].groupby("bloque")["nombre_bloque"].first().to_dict()
BLOQUES = {b: info.index[info["bloque"] == b].tolist() for b in NOMBRES}
TODAS = info.index.tolist()

I0 = int(np.searchsorted(f, pd.Timestamp(INICIO)))
PRIMERO = I0 + VENTANA_MAX - 1          # primer día con la ventana más larga completa
completo = ~np.isnan(Y).any(axis=(1, 2))

for b in BLOQUES:
    print(f"bloque {b}, {NOMBRES[b]}: {', '.join(BLOQUES[b])}")
print(f"{len(TODAS)} variables en total; {n_dias} días, del {f[0].date()} al {f[-1].date()}")

In [ ]:
def origenes(desde, hasta):
    # días t desde los que se pronostica, con toda la semana objetivo dentro del periodo
    assert pd.Timestamp(hasta) <= pd.Timestamp(TECHO), "este notebook no pasa de 2022"
    i = np.arange(n_dias)
    m = (i >= PRIMERO) & (f >= pd.Timestamp(desde)) & completo
    m &= f + pd.Timedelta(days=max(HS)) <= pd.Timestamp(hasta)
    return np.flatnonzero(m)


def pliegue(año):
    # ajuste, parada temprana (último 10% del entrenamiento) y evaluación (el año)
    corte = pd.Timestamp(f"{año - 1}-12-31")
    tr = origenes(INICIO, corte)
    n_parada = len(tr) // 10
    ajuste, parada = tr[:-n_parada - max(HS)], tr[-n_parada:]
    ev = origenes(corte + pd.Timedelta(days=1), f"{año}-12-31")
    return ajuste, parada, ev


def entradas(variables, hasta):
    # vector de cada día, estandarizado con los días de INICIO a `hasta`
    partes = [LP] + [por_dia(v) if info.loc[v, "tipo"] == "horaria" else por_dia(v)[:, :1]
                     for v in variables]
    M = np.hstack(partes)
    ref = M[I0:hasta + 1]
    assert not np.isnan(ref).any()
    M = np.clip((M - ref.mean(axis=0)) / (ref.std(axis=0) + 1e-6), -8, 8)
    return np.hstack([M, CAL_T]).astype(np.float32)


def ventanas(M, idx, ventana):
    return M[idx[:, None] - np.arange(ventana - 1, -1, -1)]


for año in AÑOS:
    aj, par, ev = pliegue(año)
    # entre ajuste, parada y evaluación no se comparte ningún precio objetivo
    assert f[aj[-1]] + pd.Timedelta(days=max(HS)) < f[par[0]]
    assert f[par[-1]] + pd.Timedelta(days=max(HS)) < f[ev[0]] + pd.Timedelta(days=min(HS))
    assert f[ev[0]].year == año and (f[ev[-1]] + pd.Timedelta(days=max(HS))).year == año
    print(f"{año}: ajuste {len(aj)} días ({f[aj[0]].date()} a {f[aj[-1]].date()}), "
          f"parada {len(par)}, evaluación {len(ev)} ({f[ev[0]].date()} a {f[ev[-1]].date()})")

## 2. Modelo

Codificador-decodificador (Sutskever et al., 2014) con celdas LSTM (Hochreiter y Schmidhuber,
1997).

- El codificador lee los últimos 28 días. Una capa densa comprime primero el vector de cada día a
  64 números, para que la red tenga casi el mismo tamaño con cualquier conjunto de variables.
- El decodificador arranca con el estado final del codificador y da 7 pasos, uno por día objetivo.
  En cada paso recibe el día de la semana y el festivo de ese día, y entrega sus 24 horas.

La pérdida es el MSE, porque el criterio de la tesis es el RMSE en logaritmo. La parada temprana
mira el último 10% del periodo de entrenamiento.

In [ ]:
def construir(n_entradas, c):
    historia = keras.Input((c["ventana"], n_entradas))
    semana = keras.Input((len(HS), CAL_K.shape[2]))
    x = layers.Dense(c["densa"], activation="relu")(historia)
    x = layers.Dropout(c["dropout"])(x)
    estados = []
    for _ in range(c["capas"]):
        x, h, s = layers.LSTM(c["unidades"], return_sequences=True, return_state=True)(x)
        estados.append([h, s])
    y = semana
    for estado in estados:
        y = layers.LSTM(c["unidades"], return_sequences=True)(y, initial_state=estado)
    y = layers.Dense(24)(layers.Dropout(c["dropout"])(y))
    modelo = keras.Model([historia, semana], y)
    modelo.compile(optimizer=keras.optimizers.Adam(learning_rate=c["tasa"]), loss="mse")
    return modelo


def pronostico(nombre, variables, año, semilla, c=FIJA, guardar_modelo=False):
    # pronóstico del año con el modelo entrenado hasta el año anterior; se guarda y no se repite
    ruta = SALIDA / "pronosticos" / f"{nombre}_{año}_s{semilla}.npy"
    if ruta.exists():
        return np.load(ruta)
    keras.backend.clear_session()
    keras.utils.set_random_seed(semilla)
    aj, par, ev = pliegue(año)
    M = entradas(variables, par[-1])
    X = lambda idx: [ventanas(M, idx, c["ventana"]), CAL_K[idx]]
    modelo = construir(M.shape[1], c)
    parar = keras.callbacks.EarlyStopping(patience=PACIENCIA, restore_best_weights=True)
    hist = modelo.fit(X(aj), Y[aj], validation_data=(X(par), Y[par]), epochs=EPOCAS,
                      batch_size=c["lote"], callbacks=[parar], verbose=0)
    pred = modelo.predict(X(ev), verbose=0)
    if guardar_modelo:
        modelo.save(ruta.with_suffix(".keras"))
    np.save(ruta, pred)
    print(f"  {nombre}, {año}, semilla {semilla}: {len(hist.history['loss'])} épocas, "
          f"pérdida de parada {min(hist.history['val_loss']):.4f}", flush=True)
    return pred


# la red de referencia: 24 horas del precio y 8 del calendario por día
construir(24 + CAL_T.shape[1], FIJA).summary()

## 3. Cómo se mide

Por cada día de pronóstico y cada horizonte se promedia el error cuadrático de las 24 horas. Los
tres años de validación se juntan y los pronósticos de las tres semillas se promedian antes de
medir. La comparación entre dos versiones es Diebold-Mariano con varianza de Newey-West
(h − 1 + 7 rezagos), igual que en XGBoost, SARIMAX y el paso 2f.

In [ ]:
def errores(nombre, variables, c=FIJA, semillas=None, guardar_modelo=False):
    partes = []
    for año in AÑOS:
        ev = pliegue(año)[2]
        pred = np.mean([pronostico(nombre, variables, año, s, c, guardar_modelo)
                        for s in (semillas or SEMILLAS)], axis=0)
        e2 = ((Y[ev] - pred) ** 2).mean(axis=2)
        i2 = (Y[ev] ** 2).mean(axis=2)           # ingenuo del día: y = 0
        partes.append(pd.DataFrame(np.hstack([e2, i2]), index=f[ev],
                                   columns=[f"e2_h{h}" for h in HS] + [f"i2_h{h}" for h in HS]))
    E = pd.concat(partes)
    E["e2_semana"] = E[[f"e2_h{h}" for h in HS]].mean(axis=1)
    E["i2_semana"] = E[[f"i2_h{h}" for h in HS]].mean(axis=1)
    return E


def media_y_ee(d, L):
    n = len(d)
    u = d - d.mean()
    v = np.mean(u * u)
    for l in range(1, min(L, n - 1) + 1):
        v += 2 * (1 - l / (L + 1)) * np.mean(u[l:] * u[:-l])
    return d.mean(), np.sqrt(v / n)


def diebold_mariano(a, b, h):
    # t negativo: a es mejor que b
    m, ee = media_y_ee((a - b).to_numpy(), h - 1 + 7)
    t = m / ee
    return t, 1 - erf(abs(t) / sqrt(2))


def rmse(E, col):
    return float(np.sqrt(E[col].mean()))


def mejoras(E):
    # mejora del RMSE frente al ingenuo del día, en %
    return {f"mejora_{k}": 100 * (1 - rmse(E, f"e2_{k}") / rmse(E, f"i2_{k}"))
            for k in ["semana"] + [f"h{h}" for h in HS_PRUEBA]}


def contra(E, E_ref):
    r = {}
    for h in HS_PRUEBA:
        r[f"t_h{h}"], r[f"p_h{h}"] = diebold_mariano(E[f"e2_h{h}"], E_ref[f"e2_h{h}"], h)
    gana = any(r[f"t_h{h}"] < 0 and r[f"p_h{h}"] < NIVEL for h in HS_PRUEBA)
    pierde = any(r[f"t_h{h}"] > 0 and r[f"p_h{h}"] < NIVEL for h in HS_PRUEBA)
    return r, gana and not pierde

## 4. Selección de bloques

La referencia es la red con solo el precio y el calendario (bloques 1 y 2). Cada bloque se agrega
a la referencia del momento y se entrena desde cero. Entra si le gana con Diebold-Mariano
p < 0,05 en h = 3, 6 o 9 y no pierde con p < 0,05 en ninguno; si entra, pasa a ser la nueva
referencia. El bloque 6 (clima) solo se prueba si entró el 4 (estado hidrológico).

In [ ]:
def seleccionar(c, prefijo, archivo):
    # selección hacia adelante de los bloques, con la configuración c
    nombre_ref, vars_ref, entraron = prefijo + "b12", [], []
    E_ref = errores(nombre_ref, vars_ref, c)
    filas = [dict(conjunto="precio y calendario", variables=0, **mejoras(E_ref))]
    for b in sorted(BLOQUES):
        if b == 6 and 4 not in entraron:
            print("El bloque 6 no se prueba porque no entró el 4")
            continue
        nombre = prefijo + "b12" + "".join(f"_{x}" for x in entraron + [b])
        vars_b = vars_ref + BLOQUES[b]
        E = errores(nombre, vars_b, c)
        prueba, entra = contra(E, E_ref)
        filas.append(dict(conjunto=f"+ {NOMBRES[b]}", variables=len(vars_b), **mejoras(E), **prueba, entra=entra))
        print(f"bloque {b}, {NOMBRES[b]}: {'entra' if entra else 'no entra'}")
        if entra:
            nombre_ref, vars_ref, E_ref = nombre, vars_b, E
            entraron.append(b)
        pd.DataFrame(filas).to_csv(SALIDA / archivo, index=False)
    print("\nBloques que entran:", ", ".join(f"{b} ({NOMBRES[b]})" for b in entraron) or "ninguno")
    return dict(nombre=nombre_ref, bloques=[1, 2] + entraron, variables=vars_ref, configuracion=c,
                E=E_ref, tabla=pd.DataFrame(filas))


primera = seleccionar(FIJA, "", "seleccion.csv")
entraron, vars_ref, E_ref = primera["bloques"][2:], primera["variables"], primera["E"]
primera["tabla"].round(3)

## 5. Prueba de control: todas las variables

Se entrena la misma red con todas las variables del panel. Sirve para dos cosas:

- comparar su error con el del conjunto elegido;
- ver si algún grupo que quedó fuera de los bloques importa. Con los modelos ya entrenados se
  desordenan entre días los datos de un grupo completo y se mide cuánto sube el RMSE de la semana
  (importancia por permutación). Se hace por grupo porque muchas variables dicen lo mismo: si se
  desordena una sola de las de lluvia, la red usa las otras y parece que ninguna importa.

Cada bloque se desordena completo, con todas las variables de los grupos de donde salió (el bloque
4, por ejemplo, con todo lo de embalses, aportes e interacciones). Los grupos que no están en
ningún bloque se desordenan uno por uno.

In [ ]:
E_todas = errores("todas", TODAS, guardar_modelo=True)
prueba, _ = contra(E_todas, E_ref)
control = pd.DataFrame([dict(conjunto="elegido", variables=len(vars_ref), **mejoras(E_ref)),
                        dict(conjunto="todas", variables=len(TODAS), **mejoras(E_todas), **prueba)])
control.to_csv(SALIDA / "control_todas.csv", index=False)
control.round(3)

In [ ]:
def posiciones(variables):
    # columnas que ocupa cada variable en el vector del día (las primeras 24 son el precio)
    pos, i = {"lp": np.arange(24)}, 24
    for v in variables:
        k = 24 if info.loc[v, "tipo"] == "horaria" else 1
        pos[v] = np.arange(i, i + k)
        i += k
    return pos


def unidades():
    # qué se desordena junto: el precio, cada bloque con sus grupos y cada grupo sin bloque
    grupos = info.groupby("grupo").groups
    de_bloque = info[info["bloque"] > 0].groupby("bloque")["grupo"].unique()
    usados = [g for gs in de_bloque for g in gs]
    assert len(usados) == len(set(usados)), "un grupo está en dos bloques"
    u = {"precio": ["lp"]}
    for b, gs in de_bloque.items():
        u[f"bloque {b}: {NOMBRES[b]}"] = [v for g in gs for v in grupos[g]]
    u.update({g: list(vs) for g, vs in grupos.items() if g not in usados})
    return u


def importancia(repeticiones=1 if PRUEBA_RAPIDA else 3):
    ruta = SALIDA / "importancia_grupos.csv"
    if ruta.exists():
        return pd.read_csv(ruta)
    pos = posiciones(TODAS)
    rng = np.random.default_rng(0)
    filas = []
    for año in AÑOS:
        aj, par, ev = pliegue(año)
        X = ventanas(entradas(TODAS, par[-1]), ev, FIJA["ventana"])
        for s in SEMILLAS:
            modelo = keras.models.load_model(SALIDA / "pronosticos" / f"todas_{año}_s{s}.keras")
            base = np.sqrt(np.mean((Y[ev] - modelo.predict([X, CAL_K[ev]], verbose=0)) ** 2))
            for nombre, lista in unidades().items():
                cols = np.concatenate([pos[v] for v in lista])
                for _ in range(repeticiones):
                    Xp = X.copy()
                    Xp[:, :, cols] = X[rng.permutation(len(ev))][:, :, cols]
                    r = np.sqrt(np.mean((Y[ev] - modelo.predict([Xp, CAL_K[ev]], verbose=0)) ** 2))
                    filas.append(dict(año=año, semilla=s, unidad=nombre, variables=len(lista),
                                      aumento=100 * (r / base - 1)))
    d = pd.DataFrame(filas)
    d.to_csv(ruta, index=False)
    return d


imp = importancia()
elegidos = {f"bloque {b}: {NOMBRES[b]}" for b in entraron}
tabla_imp = imp.groupby("unidad").agg(variables=("variables", "first"), aumento_rmse_pct=("aumento", "mean"))
tabla_imp["que_es"] = ["precio" if u == "precio" else "bloque elegido" if u in elegidos
                       else "bloque que no entró" if u.startswith("bloque") else "grupo sin bloque"
                       for u in tabla_imp.index]
tabla_imp = tabla_imp.sort_values("aumento_rmse_pct", ascending=False)
tabla_imp.round(2)

Lectura (decisión del autor): si un grupo sin bloque sube el RMSE más que el menos importante de
los bloques elegidos, se arma como bloque nuevo y se prueba con la misma regla de la sección 4. Si
ninguno lo hace, la tabla de bloques queda confirmada. Los bloques que no entraron ya se probaron
en la sección 4 y no se vuelven a probar.

In [ ]:
de_elegidos = tabla_imp.loc[tabla_imp["que_es"] == "bloque elegido", "aumento_rmse_pct"]
piso = float(de_elegidos.min()) if len(de_elegidos) else 0.0
sin_bloque = tabla_imp[tabla_imp["que_es"] == "grupo sin bloque"]
por_revisar = sin_bloque.index[sin_bloque["aumento_rmse_pct"] > piso].tolist()
print(f"Bloque elegido menos importante: {piso:.2f}% de aumento del RMSE")
print("Grupos sin bloque que lo superan:", ", ".join(por_revisar) or "ninguno")

## 6. Búsqueda corta de control

La selección se hizo con una sola configuración. Para comprobar que eso no la sesga, se prueban
10 configuraciones al azar, las mismas en dos casos: solo precio y calendario, y todas las
variables. Cada una se mide en los tres años con una semilla.

Regla (decisión del autor): la configuración fija queda respaldada en un caso si ninguna de las 10
le gana con Diebold-Mariano p < 0,05 en el error de toda la semana.

In [ ]:
def al_azar(rng):
    return dict(capas=int(rng.integers(1, 4)),
                unidades=int(rng.choice([32, 64, 128, 192, 256, 360])),
                densa=int(rng.choice([32, 64, 128, 256])),
                dropout=float(rng.choice([0.2, 0.3, 0.4, 0.5])),
                tasa=float(10 ** rng.uniform(-5, -2)),
                lote=int(rng.choice([32, 64])),
                ventana=int(rng.choice([14, 28, 56])))


rng = np.random.default_rng(0)
CONFIGURACIONES = [al_azar(rng) for _ in range(N_BUSQUEDA)]
if PRUEBA_RAPIDA:
    CONFIGURACIONES = [dict(c, capas=min(c["capas"], 2), unidades=16, densa=16) for c in CONFIGURACIONES]

filas_b = []
for caso, vars_c in [("b12", []), ("todas", TODAS)]:
    E_fija = errores(caso, vars_c, semillas=[0])
    filas_b.append(dict(caso=caso, configuracion="fija", **FIJA, rmse_semana=rmse(E_fija, "e2_semana")))
    for i, c in enumerate(CONFIGURACIONES):
        E = errores(f"busq{i:02d}_{caso}", vars_c, c, semillas=[0])
        t, p = diebold_mariano(E["e2_semana"], E_fija["e2_semana"], max(HS))
        filas_b.append(dict(caso=caso, configuracion=i, **c, rmse_semana=rmse(E, "e2_semana"),
                            t_contra_fija=t, p_contra_fija=p))
        pd.DataFrame(filas_b).to_csv(SALIDA / "busqueda_control.csv", index=False)

busqueda = pd.DataFrame(filas_b)
busqueda["le_gana_a_la_fija"] = (busqueda["t_contra_fija"] < 0) & (busqueda["p_contra_fija"] < NIVEL)
respaldo = {caso: not g["le_gana_a_la_fija"].any() for caso, g in busqueda.groupby("caso")}
for caso, g in busqueda.groupby("caso"):
    puesto = int(g["rmse_semana"].rank().loc[g["configuracion"] == "fija"].iloc[0])
    print(f"{caso}: la fija queda en el puesto {puesto} de {len(g)}; "
          f"{'respaldada' if respaldo[caso] else 'alguna le gana con p < 0,05'}")
busqueda.sort_values(["caso", "rmse_semana"]).round(4)

## 7. Selección repetida

Si alguna de las 10 configuraciones le gana a la fija con p < 0,05 en los dos casos, la fija no es
una buena base para comparar conjuntos de variables: puede castigar a los que tienen más entradas.
En ese caso la selección de la sección 4 se repite completa con esa configuración, con la misma
regla y las mismas semillas. Si le ganan varias, se usa la de menor RMSE promedio en los dos casos.

Regla (decisión del autor, tomada el 13 de noviembre de 2026 después de ver la búsqueda de control
y antes de abrir el test): la selección repetida es la que vale, y la primera se reporta al lado.
También se repite la comparación contra todas las variables.

In [ ]:
candidatas = busqueda[(busqueda["configuracion"] != "fija") & busqueda["le_gana_a_la_fija"]]
candidatas = candidatas.groupby("configuracion")["rmse_semana"].agg(["size", "mean"])
candidatas = candidatas[candidatas["size"] == 2].sort_values("mean")

if len(candidatas):
    k = int(candidatas.index[0])
    FIJA_2 = CONFIGURACIONES[k]
    print(f"La configuración {k} le gana a la fija en los dos casos:", FIJA_2, "\n")
    repetida = seleccionar(FIJA_2, "rep_", "seleccion_repetida.csv")
else:
    repetida = None
    print("Ninguna configuración le gana a la fija en los dos casos: la selección no se repite")

In [ ]:
if repetida:
    lado = primera["tabla"].set_index("conjunto")[["mejora_semana", "entra"]].join(
        repetida["tabla"].set_index("conjunto")[["mejora_semana", "entra"]],
        lsuffix=" (fija)", rsuffix=" (repetida)", how="outer")
    orden = list(dict.fromkeys(list(primera["tabla"]["conjunto"]) + list(repetida["tabla"]["conjunto"])))
    display(lado.loc[orden].round(1))

    E_todas_2 = errores("rep_todas", TODAS, FIJA_2)
    prueba, _ = contra(E_todas_2, repetida["E"])
    control_2 = pd.DataFrame([dict(conjunto="elegido", variables=len(repetida["variables"]), **mejoras(repetida["E"])),
                              dict(conjunto="todas", variables=len(TODAS), **mejoras(E_todas_2), **prueba)])
    control_2.to_csv(SALIDA / "control_todas_repetida.csv", index=False)
    display(control_2.round(3))

La primera tabla pone las dos selecciones lado a lado: la mejora en la semana, en porcentaje, y si
el bloque entra con cada configuración. La segunda compara el conjunto elegido en la repetición con
todas las variables, las dos con la configuración nueva.

## 8. Resultado

Queda guardado en `seleccion.json`: los bloques y variables que entran, la configuración con la que
se eligieron y si la búsqueda de control respalda a la fija. Si la selección se repitió, lo que
queda como resultado es la repetida, y la primera se guarda aparte. Con esto sigue la etapa 2 del plan, la
búsqueda bayesiana de hiperparámetros sobre los bloques elegidos, y después el test.

La selección usa 2020-2022, que ya se había usado para elegir las variables del paso 2f y las
configuraciones de XGBoost y SARIMAX. Por eso se reporta como exploratoria; el test sí queda
limpio, porque todo se decide antes de abrirlo.

In [ ]:
final = repetida or primera
resultado = dict(bloques=final["bloques"], variables=final["variables"], configuracion=final["configuracion"],
                 nombre=final["nombre"], semillas=SEMILLAS, años_validacion=AÑOS, mejoras=mejoras(final["E"]),
                 grupos_por_revisar=por_revisar, configuracion_respaldada=respaldo,
                 seleccion_repetida=repetida is not None,
                 primera_seleccion=dict(bloques=primera["bloques"], variables=primera["variables"],
                                        configuracion=FIJA, mejoras=mejoras(primera["E"])),
                 prueba_rapida=PRUEBA_RAPIDA)
json.dump(resultado, open(SALIDA / "seleccion.json", "w", encoding="utf-8"), indent=2, ensure_ascii=False)
print(json.dumps(resultado, indent=2, ensure_ascii=False))